# LambdaRank, LambdaMART, and Listwise Objectives

The narrative companion to `lambdarank_lambdamart_listwise.py`, which **owns every number** — this
notebook imports it and walks the topic section by section. The predecessor (`learning-to-rank-pairwise`)
reduced ranking to a pairwise logistic surrogate (RankNet) whose gradient factorizes into a per-document
force $\lambda_i$. That force is **position-blind**: a swap at ranks 1–2 and a swap at ranks 99–100 carry
the same magnitude. Here we make the objective position-aware three ways and stay honest about what each
one *is*:

- **LambdaRank** scales each pair force by the $|\Delta\text{NDCG}_{ij}|$ a swap would cause — but the
  resulting field is **not the gradient of any scalar loss** (globally).
- **LambdaMART** boosts those $\lambda$ into regression trees, buying a nonlinear scorer.
- **Listwise** (ListNet / ListMLE) replaces the heuristic with a proper **convex** Plackett–Luce loss.

The substrate, the RankNet machinery, the NDCG closed forms, and the metrics are all **imported** — we
re-derive nothing.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import lambdarank_lambdamart_listwise as lr

corpus = lr._corpus()
print("docs", corpus["n_docs"], "queries", corpus["n_queries"], "TOPK", lr.TOPK)
print("train queries", len(corpus["train_q"]), "/ test queries", len(corpus["test_q"]))

## 1 — The $\Delta$NDCG closed form (the LambdaRank weight)

Swapping the documents at 1-indexed ranks $p,q$ changes DCG by $(G(g_p)-G(g_q))(D(q)-D(p))$, where
$G$ is the gain and $D$ the discount. Since the ideal DCG depends only on the grade *multiset* — which a
permutation leaves unchanged — the normalizer is swap-invariant, so

$$|\Delta\text{NDCG}_{ij}| = \frac{|G(g_i)-G(g_j)|\,|D(r_i)-D(r_j)|}{\text{IDCG}}.$$

Using the **untruncated** ideal DCG makes the two-term identity exact for *every* pair (including ones
that straddle the top-$k$ cutoff). The closed form matches a physical-swap recomputation to machine
precision.

In [ ]:
w = lr.fit_lambdarank(corpus, corpus["train_q"])
q = lr.pick_worked_query(corpus, w)
ranking = lr.ranking_from_w(corpus, q, w)
grades_q = corpus["grades"][q]
for (p, qq) in [(1, 2), (2, 3), (1, 10), (9, 11), (5, 50)]:
    cf = lr.delta_ndcg_swap(grades_q, ranking, p, qq)
    br = lr.delta_ndcg_brute(grades_q, ranking, p, qq)
    print(f"ranks ({p:2d},{qq:3d})  closed-form={cf:.6f}  brute={br:.6f}  |diff|={abs(cf-br):.2e}")

## 2 — LambdaRank = RankNet, reweighted (and where the force goes)

Forcing every $|\Delta\text{NDCG}_{ij}|\equiv 1$ recovers the imported `lambda_forces` bit-for-bit — the
collapse anchor that makes LambdaRank a *reweighting* of RankNet, not a new object. Because the discount
marginal $D(p)-D(p{+}1)$ is steep at the head and flat in the tail, the $|\Delta\text{NDCG}|$ weight is
**strictly decreasing in the pair's ranks**, so LambdaRank concentrates its gradient mass at the top of
the list where RankNet spreads it by pair count.

In [ ]:
# collapse anchor: uniform-weight LambdaRank == lambda_forces
uni = lr.lambdarank_forces(w, corpus["feats"][q], corpus["y"][q], grades_q, uniform=True)
ref = lr.lambda_forces(w, corpus["feats"][q], corpus["y"][q])
print("uniform-weight LambdaRank == lambda_forces:", np.allclose(uni, ref, atol=1e-12))

wd = lr.weight_decreasing_in_rank()
print(f"|dNDCG| weight  head(1,2)={wd['head_1_2']:.4f}  tail(9,10)={wd['tail_9_10']:.4f}")

gc = lr.gradient_concentration_by_rank(corpus, q, w)
print(f"top-3 gradient share  RankNet={gc['ranknet_top3_share']:.3f}  "
      f"LambdaRank={gc['lambdarank_top3_share']:.3f}")

## 3 — The catch: LambdaRank optimizes no scalar loss

A vector field is the gradient of a scalar potential iff its Jacobian $\partial\lambda_i/\partial s_j$ is
symmetric (Clairaut). RankNet's $\lambda$ *is* $-\nabla L_{\text{RankNet}}$, so its Jacobian is symmetric
everywhere. LambdaRank's weights depend on the current ranking, so **within** a no-swap cell the weights
are constant and the field is locally a (weighted-RankNet) gradient — its Jacobian is symmetric there
too. The obstruction is **global**: across a swap, a *spectator* pair's weight jumps, so the field is
**discontinuous** and no $C^1$ potential exists. We see it three ways on a 3-document toy: the within-cell
Jacobians are symmetric, the spectator force jumps by a fixed amount across the swap, and the loop
integral $\oint\lambda\cdot d\mathbf{s}$ is zero for RankNet but nonzero for LambdaRank.

In [ ]:
toy = lr._toy_query()
rn, lrf = lr._toy_fields(toy)
s = toy["s_base"]
Jr, Jl = lr.numerical_jacobian(rn, s), lr.numerical_jacobian(lrf, s)
print("within-cell Jacobian asymmetry  RankNet=%.2e  LambdaRank=%.2e"
      % (np.max(np.abs(Jr - Jr.T)), np.max(np.abs(Jl - Jl.T))))
print("spectator jump across swap     RankNet=%.2e  LambdaRank=%.4f"
      % (lr.swap_discontinuity_witness(rn, s, 0, 1, eps=1e-5),
         lr.swap_discontinuity_witness(lrf, s, 0, 1, eps=1e-5)))
print("loop circulation               RankNet=%.4f  LambdaRank=%.4f"
      % (lr.closed_loop_circulation(rn, s, 0, 1), lr.closed_loop_circulation(lrf, s, 0, 1)))

## 4 — Listwise objectives: a proper, convex loss

ListMLE is the negative log-likelihood of the Plackett–Luce model given the ideal permutation $\pi^*$:

$$L_{\text{ListMLE}}(s) = -\log P(\pi^*\mid s) = \sum_r\Big[\operatorname{logsumexp}(s_{\text{suffix}}) - s_{\pi^*(r)}\Big],$$

a sum of (log-sum-exp $-$ linear) terms, hence **convex** in $s$ (with a single global-shift null
direction). ListNet's top-one cross-entropy is convex too. Both have FD-checked gradients and a PSD
Hessian; ListMLE $\to 0$ as a perfectly-ordered score is scaled up (a limit, not a finite equality).
This is the principled contrast to LambdaRank's non-integrable field.

In [ ]:
from scipy.optimize import check_grad
docs = lr.candidate_set(corpus, q)
feat_cand = corpus["feats"][q][docs]
pos = {d: i for i, d in enumerate(docs)}
perm_local = [pos[d] for d in lr.optimal_permutation(corpus, q, docs)]
target_p = lr.listnet_top1_target(corpus, q, docs)
print("ListMLE grad FD error:", check_grad(lr.listmle_loss, lr.listmle_grad,
                                            np.array([0.3, -0.2, 0.5]), feat_cand, perm_local))
print("ListNet grad FD error:", check_grad(lr.listnet_loss, lr.listnet_grad,
                                            np.array([0.3, -0.2, 0.5]), feat_cand, target_p))
wm = np.array([0.2, -0.3, 0.4])
print("ListMLE Hessian min eig:", float(np.linalg.eigvalsh(lr.listmle_hessian(wm, feat_cand, perm_local)).min()))
print("ListMLE -> 0 at scale 20:", lr.listmle_loss_scores(20.0 * np.arange(6, 0, -1.0), list(range(6))))

## 5 — LambdaMART: boosting $\lambda$ into trees

LambdaMART feeds the per-document LambdaRank $\lambda$ as pseudo-residuals to gradient-boosted regression
trees: each round fits a tree to $-\lambda$ and adds $\nu\,h_t$ to the scores. The trees buy a **nonlinear**
scorer the linear RankNet cannot express. On a constructed XOR-interaction query — relevance is high iff
two features *agree* — every linear scorer is rank-capped below 1, while a depth-2 tree reaches NDCG = 1.

In [ ]:
d = lr.constructed_nonlinear_toy()
print(f"XOR instance  best-linear NDCG={d['ndcg_best_linear']:.3f}  "
      f"LambdaMART NDCG={d['ndcg_lambdamart']:.3f}")
print("boosting curve (round, test NDCG@10):")
for rnd, nd in lr.boost_curve(corpus):
    print(f"  {rnd:3d}  {nd:.4f}")

## 6 — The method comparison

All five learned scorers, evaluated on the held-out test queries, against RRF and the best single leg.
Every learned method clearly beats RRF and each single leg; among themselves they cluster *within the
confidence interval* — the honest verdict on this forgiving corpus. The seed-free wins are structural
(the top-concentration of LambdaRank's gradient, the XOR ceiling LambdaMART escapes), not the aggregate
NDCG deltas.

In [ ]:
res = lr.method_comparison(corpus)
print(f"{'method':12s}  {'NDCG@10':>8s}  {'recall@10':>9s}")
for m in ("ranknet", "lambdarank", "listnet", "listmle", "lambdamart", "rrf", "best_leg"):
    v = res["test"][m]
    print(f"{m:12s}  {v['ndcg']:8.4f}  {v['recall']:9.4f}")
hci = res["ci"][res["winner"]]
print(f"\nwinner={res['winner']}  best_leg={res['best_leg']}  "
      f"CI=[{hci['ci_lo']:.3f}, {hci['ci_hi']:.3f}] (n={hci['n']})")

## Closing

Pairwise (a loss, position-blind) $\to$ LambdaRank (position-aware, **not** a global loss) $\to$ listwise
(a proper **convex** loss). LambdaRank is a *heuristic gradient*: locally a weighted-RankNet potential,
globally non-integrable, yet it empirically ascends NDCG (Donmez–Svore–Burges). LambdaMART trades the
linear scorer for boosted trees; ListNet/ListMLE recover a genuine likelihood. On this synthetic
complementary-view corpus the methods cluster within sampling noise — the provable separations are
structural. The next step, listwise *LLM* rerankers (RankGPT), scores whole permutations with a language
model.